# N4 v2 — Calibrated duplicate and leakage audit of public agricultural image datasets (CPU only)
What changed from v1 (lessons from the PlantVillage run):
* **Mirror copies are skipped.** Some Kaggle uploads ship the same class folders twice (e.g. `color/` and
  `plantvillage dataset/color/`); v1 counted both and reported 100 % duplicates.
* **The Hamming threshold is calibrated per dataset, not reused from WEAPD.** On lab images with a uniform
  background, pHash distance <= 8 matches *different* leaves. v2 collects candidate pairs up to distance 10,
  verifies each with pixel-level normalised cross-correlation (NCC) of 48x48 thumbnails, and writes
  contact sheets per distance band so the threshold can be chosen by eye.
* The summary is reported for several rules (MD5 only, pHash <= 2/4/6/8, verified = pHash <= 10 and NCC >= 0.90).

**Before running:** Accelerator **None (CPU)** · Add Input → PlantVillage (abdallahalidev/plantvillage-dataset)
and PlantDoc (and any other crop dataset). Download `n4v2_audit_small.zip` from the Output tab.

In [ ]:
CFG = dict(
    CAND_DIST=10,                                 # collect candidate pairs up to this pHash distance
    RULE_DISTS=[2, 4, 6, 8],                      # report the audit at these pure-pHash thresholds
    NCC_VERIFY=0.90,                              # "verified" rule: pHash <= CAND_DIST and thumbnail NCC >= this
    THUMB=48,
    CALIB_BINS=[(0, 0), (1, 2), (3, 4), (5, 6), (7, 8), (9, 10)],
    PAIRS_PER_BIN=20,
    EXCLUDE_DIRS=["grayscale", "segmented"],      # PlantVillage: audit only "color"
    MIN_CLASSES=2, WORKERS=4, SEED=0,
    INPUT="/kaggle/input", OUT="/kaggle/working/n4v2_audit",
)
import os, sys, hashlib, time
import numpy as np, pandas as pd
from PIL import Image, ImageFile, ImageDraw
from multiprocessing import Pool
ImageFile.LOAD_TRUNCATED_IMAGES = True
os.makedirs(CFG["OUT"], exist_ok=True)
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")
print(CFG)

In [ ]:
# 1. Find class roots, then drop mirror copies (same class folders with the same file names)
def has_images(d, k=30):
    try: return any(f.lower().endswith(IMG_EXT) for f in os.listdir(d)[:k])
    except Exception: return False

roots = []
for d, subdirs, _ in os.walk(CFG["INPUT"], followlinks=True):
    if any(x.lower() in CFG["EXCLUDE_DIRS"] for x in d.split(os.sep)): continue
    img_sub = [s for s in subdirs if has_images(os.path.join(d, s))]
    if len(img_sub) >= CFG["MIN_CLASSES"] and not has_images(d):
        roots.append(d)
roots = [r for r in roots if not any(o != r and o.startswith(r + os.sep) for o in roots)]
def dataset_of(r):
    p = r.split(os.sep); i = p.index("input")
    if p[i + 1] == "datasets" and len(p) > i + 3: return p[i + 3]        # /kaggle/input/datasets/<owner>/<slug>/...
    if p[i + 1] == "competitions" and len(p) > i + 2: return p[i + 2]
    return p[i + 1]
SPLIT_WORDS = {"train": "train", "training": "train", "test": "test", "testing": "test", "val": "val", "valid": "val", "validation": "val"}
def split_of(r):
    for part in reversed(r.lower().split(os.sep)):
        if part in SPLIT_WORDS: return SPLIT_WORDS[part]
    return "all"
def signature(r):
    s = []
    for c in sorted(os.listdir(r)):
        cd = os.path.join(r, c)
        if os.path.isdir(cd):
            fs = sorted(f for f in os.listdir(cd) if f.lower().endswith(IMG_EXT))
            s.append((c, len(fs), tuple(fs[:5]), tuple(fs[-5:])))
    return tuple(s)
seen, kept = {}, []
for r in sorted(roots, key=len):
    key = (dataset_of(r), split_of(r), signature(r))
    if key in seen:
        print("SKIP mirror copy:", r, "\n   same as:", seen[key]); continue
    seen[key] = r; kept.append(r)
roots = kept
for r in roots: print(dataset_of(r), "|", split_of(r), "|", r)

In [ ]:
# 2. Hash every image: MD5 of the file, 64-bit DCT pHash, and a 48x48 grey thumbnail for NCC verification
_N = 32
_C = np.array([[np.cos(np.pi * (2 * x + 1) * u / (2 * _N)) for x in range(_N)] for u in range(_N)])
def hash_one(p):
    try:
        raw = open(p, "rb").read(); im = Image.open(p); im.load()
        g = im.convert("L")
        a = np.asarray(g.resize((_N, _N), Image.LANCZOS), dtype=np.float64)
        low = (_C @ a @ _C.T)[:8, :8].flatten()
        bits = low > np.median(low[1:])
        th = np.asarray(g.resize((CFG["THUMB"], CFG["THUMB"]), Image.BILINEAR), dtype=np.uint8).ravel()
        return hashlib.md5(raw).hexdigest(), int("".join("1" if b else "0" for b in bits), 2), im.size[0], im.size[1], th
    except Exception:
        return None
items = []
for r in roots:
    for c in sorted(os.listdir(r)):
        cd = os.path.join(r, c)
        if not os.path.isdir(cd): continue
        for f in os.listdir(cd):
            if f.lower().endswith(IMG_EXT): items.append((dataset_of(r), split_of(r), c, os.path.join(cd, f)))
print(len(items), "images"); t0 = time.time()
with Pool(CFG["WORKERS"]) as pool:
    hs = pool.map(hash_one, [x[3] for x in items], chunksize=64)
ok = [h is not None for h in hs]
df = pd.DataFrame([dict(dataset=d, split=s, label=c, path=p, md5=h[0], phash=h[1], w=h[2], h=h[3])
                   for (d, s, c, p), h in zip(items, hs) if h is not None])
THUMBS = np.stack([h[4] for h in hs if h is not None]).astype(np.float32)
Z = THUMBS - THUMBS.mean(1, keepdims=True)
Z /= (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-6)          # unit vectors -> dot product = NCC
del THUMBS, hs
print(f"hashed {len(df)} images in {(time.time()-t0)/60:.1f} min ({len(items)-len(df)} unreadable)")
for name, g in df.groupby("dataset"):
    share = g.md5.duplicated().mean()
    print(f"{name}: exact-copy share {100*share:.1f}%" + ("   <-- WARNING: check for a mirrored folder" if share > 0.4 else ""))

In [ ]:
# 3. Candidate pairs up to CAND_DIST with a pigeonhole index (CAND_DIST+1 bit-segments guarantee no pair is missed),
#    then NCC verification of every candidate pair
def make_segs(T):
    k = T + 1; base, extra = divmod(64, k); segs, off = [], 0
    for i in range(k):
        ln = base + (1 if i < extra else 0); segs.append((off, ln)); off += ln
    return segs
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def near_pairs(codes, T):
    codes = np.asarray(codes, dtype=np.uint64); out = {}
    for off, ln in make_segs(T):
        key = (codes >> np.uint64(off)) & np.uint64((1 << ln) - 1)
        order = np.argsort(key, kind="stable"); ks = key[order]
        starts = np.flatnonzero(np.r_[True, ks[1:] != ks[:-1]]); ends = np.r_[starts[1:], len(ks)]
        for a, b in zip(starts, ends):
            if b - a < 2: continue
            idx = order[a:b]; cb = codes[idx]
            for s0 in range(0, len(idx), 1000):
                blk = cb[s0:s0 + 1000]
                x = (blk[:, None] ^ cb[None, :]).view(np.uint8).reshape(len(blk), len(idx), 8)
                d = POP[x].sum(2, dtype=np.uint8)
                ii, jj = np.nonzero(d <= T); dd = d[ii, jj]; ii = ii + s0
                keep = jj > ii
                for i, j, v in zip(idx[ii[keep]], idx[jj[keep]], dd[keep]):
                    out[(int(min(i, j)), int(max(i, j)))] = int(v)
    return out

def audit(g, pi, pj, name, rule):
    n = len(g); par = np.arange(n)
    def find(i):
        while par[i] != i: par[i] = par[par[i]]; i = par[i]
        return i
    first = {}
    for i, m in enumerate(g.md5.values):
        if m in first: par[find(i)] = find(first[m])
        else: first[m] = i
    for i, j in zip(pi, pj): par[find(i)] = find(j)
    grp = np.array([find(i) for i in range(n)])
    gs = pd.Series(grp); vc = gs.value_counts(); size = gs.map(vc).values
    nlab = gs.map(pd.DataFrame({"g": grp, "l": g.label.values}).groupby("g").l.nunique()).values
    in_group = int((size > 1).sum()); n_groups = int((vc > 1).sum())
    row = dict(dataset=name, rule=rule, images=n, classes=g.label.nunique(), pairs=len(pi),
               dup_groups=n_groups, images_in_dup_groups=in_group, pct_images_in_dup_groups=round(100 * in_group / n, 2),
               redundant_copies=in_group - n_groups, pct_redundant=round(100 * (in_group - n_groups) / n, 2),
               images_in_label_conflict=int((nlab > 1).sum()))
    sp = g.split.values
    if {"train", "test"} <= set(sp):
        trg = set(grp[sp == "train"]); te = grp[sp == "test"]
        row["pct_official_test_leaked"] = round(100 * np.mean([x in trg for x in te]), 2)
    rng = np.random.default_rng(CFG["SEED"]); leaks = []
    for _ in range(5):
        is_test = rng.random(n) < 0.2
        trg = set(grp[~is_test]); leaks.append(np.mean([x in trg for x in grp[is_test]]))
    row["pct_test_leaked_naive_random_split"] = round(100 * float(np.mean(leaks)), 2)
    return row, grp

rows, calib, all_pairs = [], [], {}
for name, g in df.groupby("dataset"):
    gidx = g.index.values; g = g.reset_index(drop=True); t0 = time.time()
    P = near_pairs(g.phash.values, CFG["CAND_DIST"])
    pr = pd.DataFrame([(i, j, d) for (i, j), d in P.items()], columns=["i", "j", "dist"])
    if len(pr):
        zi, zj = Z[gidx[pr.i.values]], Z[gidx[pr.j.values]]
        pr["ncc"] = np.einsum("ij,ij->i", zi, zj).round(4)
    else:
        pr["ncc"] = []
    pr["label_i"] = g.label.values[pr.i.values]; pr["label_j"] = g.label.values[pr.j.values]
    pr["cross_label"] = pr.label_i != pr.label_j
    print(f"{name}: {len(pr)} candidate pairs <= {CFG['CAND_DIST']} in {(time.time()-t0)/60:.1f} min")
    for lo, hi in CFG["CALIB_BINS"]:
        b = pr[(pr.dist >= lo) & (pr.dist <= hi)]
        calib.append(dict(dataset=name, dist_band=f"{lo}-{hi}", pairs=len(b),
                          pct_cross_label=round(100 * b.cross_label.mean(), 1) if len(b) else None,
                          median_ncc=round(b.ncc.median(), 3) if len(b) else None,
                          pct_ncc_ge_verify=round(100 * (b.ncc >= CFG["NCC_VERIFY"]).mean(), 1) if len(b) else None))
    rules = [("md5_only", pr.iloc[:0])] + [(f"phash_le_{T}", pr[pr.dist <= T]) for T in CFG["RULE_DISTS"]] + \
            [(f"verified_le_{CFG['CAND_DIST']}_ncc_{CFG['NCC_VERIFY']}", pr[pr.ncc >= CFG["NCC_VERIFY"]])]
    for rule, sel in rules:
        row, grp = audit(g, sel.i.values, sel.j.values, name, rule); rows.append(row)
        if rule.startswith("verified"): g["group_verified"] = grp
    g[["dataset", "split", "label", "path", "md5", "phash", "group_verified"]].assign(phash=lambda d: d.phash.map(lambda v: f"{int(v):016x}")) \
        .to_csv(os.path.join(CFG["OUT"], f"hashes_{name}.csv"), index=False)
    pr.assign(path_i=g.path.values[pr.i.values], path_j=g.path.values[pr.j.values]) \
        .to_csv(os.path.join(CFG["OUT"], f"pairs_{name}.csv.gz"), index=False)
    all_pairs[name] = (g, pr)
summary = pd.DataFrame(rows); summary.to_csv(os.path.join(CFG["OUT"], "n4v2_audit_summary.csv"), index=False)
calib = pd.DataFrame(calib); calib.to_csv(os.path.join(CFG["OUT"], "n4v2_calibration_by_distance.csv"), index=False)
print(calib.to_string()); summary

In [ ]:
# 4. Contact sheets per distance band (left|right, caption = distance, NCC, labels; red caption = different classes)
W = 150
for name, (g, pr) in all_pairs.items():
    for lo, hi in CFG["CALIB_BINS"]:
        b = pr[(pr.dist >= lo) & (pr.dist <= hi)]
        if not len(b): continue
        s = b.sample(min(CFG["PAIRS_PER_BIN"], len(b)), random_state=CFG["SEED"])
        H = W + 22; cols = 2; per_col = (len(s) + cols - 1) // cols
        sheet = Image.new("RGB", (cols * (2 * W + 30), per_col * H), "white"); dr = ImageDraw.Draw(sheet)
        for k, r in enumerate(s.itertuples()):
            x0 = (k // per_col) * (2 * W + 30); y0 = (k % per_col) * H
            for jj, p in enumerate([g.path.values[r.i], g.path.values[r.j]]):
                try: sheet.paste(Image.open(p).convert("RGB").resize((W, W)), (x0 + jj * (W + 4), y0))
                except Exception: pass
            cap = f"#{k} d={r.dist} ncc={r.ncc:.2f} {r.label_i[:14]}|{r.label_j[:14]}"
            dr.text((x0, y0 + W + 4), cap, fill=(200, 0, 0) if r.cross_label else (0, 0, 0))
        sheet.save(os.path.join(CFG["OUT"], f"calib_{name}_d{lo}-{hi}.jpg"), quality=85)
    # NCC-verified pairs with the largest distance (the hardest cases for the verified rule)
    v = pr[pr.ncc >= CFG["NCC_VERIFY"]].sort_values("dist", ascending=False).head(CFG["PAIRS_PER_BIN"])
    if len(v):
        sheet = Image.new("RGB", (2 * W + 30, len(v) * (W + 22)), "white"); dr = ImageDraw.Draw(sheet)
        for k, r in enumerate(v.itertuples()):
            for jj, p in enumerate([g.path.values[r.i], g.path.values[r.j]]):
                try: sheet.paste(Image.open(p).convert("RGB").resize((W, W)), (jj * (W + 4), k * (W + 22)))
                except Exception: pass
            dr.text((0, k * (W + 22) + W + 4), f"#{k} d={r.dist} ncc={r.ncc:.2f} {r.label_i[:14]}|{r.label_j[:14]}",
                    fill=(200, 0, 0) if r.cross_label else (0, 0, 0))
        sheet.save(os.path.join(CFG["OUT"], f"calib_{name}_verified_hardest.jpg"), quality=85)
print(sorted(os.listdir(CFG["OUT"])))

In [ ]:
# 5. Per-class share of images with a copy under the verified rule
pc = []
for name, (g, pr) in all_pairs.items():
    sz = g.group_verified.map(g.group_verified.value_counts())
    t = g.assign(dup=sz > 1).groupby("label").dup.agg(["sum", "count"]); t["pct"] = (100 * t["sum"] / t["count"]).round(1)
    t.insert(0, "dataset", name); pc.append(t.reset_index())
pc = pd.concat(pc) if pc else pd.DataFrame(); pc.to_csv(os.path.join(CFG["OUT"], "n4v2_perclass_verified.csv"), index=False)
pc.sort_values("pct", ascending=False).head(25) if len(pc) else pc

In [ ]:
# 6. Small zip for download (no images except the contact sheets)
import zipfile
zp = "/kaggle/working/n4v2_audit_small.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(CFG["OUT"]): z.write(os.path.join(CFG["OUT"], f), f)
print(zp, "%.1f MB" % (os.path.getsize(zp) / 1e6))